# NLP — Notícias Motor1 Brasil

NLP, ou Processamento de Linguagem Natural, é uma área da ciência de dados
que estuda como transformar textos em informação estruturada. Com essas
técnicas, conseguimos analisar notícias automotivas, identificar temas,
contar palavras, comparar documentos e preparar dados textuais para os
modelos de aprendizado de máquina da próxima etapa do projeto.

In [1]:
import os
from pathlib import Path

import pandas as pd

# Caminho relativo ao notebook (não depende de máquina/usuário):
# este notebook está em projeto/notebooks/pln/, e os dados ficam em projeto/dados/,
# dois níveis acima — mesma convenção usada nos demais notebooks do projeto.
_NOTEBOOK_DIR = Path(os.path.abspath("__file__")).parent
pasta_dados = (_NOTEBOOK_DIR / ".." / ".." / "dados").resolve()
CAMINHO_DADOS = pasta_dados / "noticias_motor1_2016_2026.csv"

df = pd.read_csv(CAMINHO_DADOS)

print(f"Antes: {len(df)} noticias")

df = df.drop_duplicates(subset="url").reset_index(drop=True)

print(f"Depois: {len(df)} noticias")

df.head()

Antes: 26917 noticias
Depois: 26917 noticias


,id_noticia,data_publicacao,ano,mes,titulo,subtitulo,texto_completo,categoria,autor,url
0,121597,2016-01-01 08:01:33,2016,1,Faróis a laser: você está disposto a pagar (ca...,Faróis a laser: você está disposto a pagar (ca...,Top Videos: Queremos a sua opinião! O que você...,Geral,Por:Redação,https://motor1.uol.com.br/news/121597/farois-a...
1,121653,2016-01-04 15:18:00,2016,1,Inusitada: BMW R1200 GS ganha versão com traçã...,Inusitada: BMW R1200 GS ganha versão com traçã...,Queremos a sua opinião! O que você gostaria de...,Motos,Por:Redação2,https://motor1.uol.com.br/news/121653/inusitad...
2,121707,2016-01-05 10:00:00,2016,1,FCA apresenta central Uconnect com Android Aut...,FCA apresenta central Uconnect com Android Aut...,Queremos a sua opinião! O que você gostaria de...,Tecnologia,Por:Redação,https://motor1.uol.com.br/news/121707/fca-apre...
3,121745,2016-01-06 13:45:24,2016,1,CES 2016: BMW apresenta capacete com head-up d...,CES 2016: BMW apresenta capacete com head-up d...,Queremos a sua opinião! O que você gostaria de...,Motos,Por:Redação,https://motor1.uol.com.br/news/121745/ces-2016...
4,121766,2016-01-09 08:20:25,2016,1,Veja a lista dos 10 carros mais vendidos nos E...,Veja a lista dos 10 carros mais vendidos nos E...,Top Videos: Queremos a sua opinião! O que você...,Geral,Por:Dyogo Fagundes,https://motor1.uol.com.br/news/121766/veja-a-l...


In [2]:
df["texto_completo"].iloc[0]

'Top Videos: Queremos a sua opinião! O que você gostaria de ver no Motor1.com? - Equipe do Motor1.com'

## Trabalhando apenas com o texto

Por enquanto, vamos trabalhar somente com a coluna `texto_completo`, que
contém o corpo completo de cada notícia. As outras colunas, como `titulo`,
`subtitulo`, `categoria`, `autor` e `data_publicacao`, continuam no
`DataFrame`, mas vamos deixar elas de lado neste primeiro momento para
entender melhor o conteúdo textual.

## Passos da análise

Vamos preparar os textos aos poucos:

1. Remover textos padrão do site (boilerplate de scraping).
2. Limpar os textos.
3. Remover palavras muito comuns.
4. Criar uma representação Bag of Words.
5. Contar palavras frequentes.
6. Transformar textos em números para análises posteriores.

## 1. Removendo textos padrão do site (boilerplate)

Antes da limpeza linguística, vale a pena inspecionar o texto coletado.
Boa parte das notícias carrega um trecho fixo do site, que não é conteúdo
editorial da matéria, e sim um widget de enquete presente na página:

> "Queremos a sua opinião! O que você gostaria de ver no Motor1.com? -
> Equipe do Motor1.com"

Isso é um artefato comum de web scraping: o seletor usado para capturar o
corpo da notícia também capturou um elemento de interface do site. Se não
removermos esse trecho, ele aparece em quase todas as notícias e polui a
contagem de palavras frequentes no Bag of Words.

Na célula abaixo:

- `re.sub` substitui os trechos que casam com o padrão por uma string
  vazia.
- Os padrões cobrem o texto da enquete e o prefixo `"Top Videos:"`, que
  aparece em parte dos registros.
- `.apply(remover_boilerplate)` aplica a função em todas as notícias.

Exemplo: `"Top Videos: Queremos a sua opinião!... Equipe do Motor1.com"`
vira `""`.

In [3]:
import re

PADROES_BOILERPLATE = [
    r"Top Videos:\s*",
    r"Queremos a sua opini[aã]o!.*?Equipe do Motor1\.com",
]


def remover_boilerplate(texto):
    texto_limpo = str(texto)
    for padrao in PADROES_BOILERPLATE:
        texto_limpo = re.sub(padrao, "", texto_limpo, flags=re.IGNORECASE)
    return texto_limpo.strip()


df["texto_sem_boilerplate"] = df["texto_completo"].apply(remover_boilerplate)

df[["texto_completo", "texto_sem_boilerplate"]].head()

,texto_completo,texto_sem_boilerplate
0,Top Videos: Queremos a sua opinião! O que você...,
1,Queremos a sua opinião! O que você gostaria de...,
2,Queremos a sua opinião! O que você gostaria de...,
3,Queremos a sua opinião! O que você gostaria de...,
4,Top Videos: Queremos a sua opinião! O que você...,


## 2. Limpeza básica

Na célula abaixo:

- `wordpunct_tokenize` separa o texto em palavras e pontuação.
- `texto.lower()` coloca tudo em minúsculas.
- `unidecode(texto)` troca letras acentuadas por letras sem acento.
- `token.isalnum()` mantém apenas letras e números.
- `" ".join(tokens)` junta os tokens em uma frase limpa.
- `.apply(limpar_texto)` aplica a função em todas as notícias.

Exemplo: `"Faróis a laser!"` vira `"farois a laser"`.

In [4]:
from nltk.tokenize import wordpunct_tokenize
from unidecode import unidecode

In [5]:
texto_exemplo = df["texto_sem_boilerplate"].iloc[0].lower()
texto_exemplo = unidecode(texto_exemplo)
texto_exemplo = wordpunct_tokenize(texto_exemplo)
texto_exemplo

[]

In [6]:
def limpar_texto(texto):
    texto = texto.lower()
    texto = unidecode(texto)
    tokens = wordpunct_tokenize(texto)
    tokens = [token for token in tokens if token.isalnum()]
    return " ".join(tokens)


df["texto_limpo"] = df["texto_sem_boilerplate"].apply(limpar_texto)

df[["texto_sem_boilerplate", "texto_limpo"]].iloc[0]

texto_sem_boilerplate    
texto_limpo              
Name: 0, dtype: str

## 3. Removendo stopwords

Stopwords são palavras muito comuns, como `a`, `o`, `de`, `para` e `que`.
Elas aparecem muito, mas geralmente ajudam pouco a entender o tema de uma
notícia.

Na célula abaixo:

- `stopwords.words("portuguese")` carrega stopwords em português.
- `texto.split()` separa o texto limpo em palavras.
- `token not in stopwords_pt` remove as palavras muito comuns.
- `.str.join(" ")` junta os tokens restantes em um texto sem stopwords.
- `.apply(remover_stopwords)` aplica a função em todas as notícias.

Exemplo: `"o carro tem um motor turbo"` vira `["carro", "motor", "turbo"]`.

In [7]:
import nltk

from nltk.corpus import stopwords


nltk.download("stopwords", quiet=True)

stopwords_pt = stopwords.words("portuguese")
stopwords_pt = [unidecode(palavra) for palavra in stopwords_pt]
stopwords_pt = set(stopwords_pt)

In [8]:
stopwords_pt

{'a',
 'ao',
 'aos',
 'aquela',
 'aquelas',
 'aquele',
 'aqueles',
 'aquilo',
 'as',
 'ate',
 'com',
 'como',
 'da',
 'das',
 'de',
 'dela',
 'delas',
 'dele',
 'deles',
 'depois',
 'do',
 'dos',
 'e',
 'ela',
 'elas',
 'ele',
 'eles',
 'em',
 'entre',
 'era',
 'eram',
 'eramos',
 'essa',
 'essas',
 'esse',
 'esses',
 'esta',
 'estamos',
 'estao',
 'estar',
 'estas',
 'estava',
 'estavam',
 'estavamos',
 'este',
 'esteja',
 'estejam',
 'estejamos',
 'estes',
 'esteve',
 'estive',
 'estivemos',
 'estiver',
 'estivera',
 'estiveram',
 'estiveramos',
 'estiverem',
 'estivermos',
 'estivesse',
 'estivessem',
 'estivessemos',
 'estou',
 'eu',
 'foi',
 'fomos',
 'for',
 'fora',
 'foram',
 'foramos',
 'forem',
 'formos',
 'fosse',
 'fossem',
 'fossemos',
 'fui',
 'ha',
 'haja',
 'hajam',
 'hajamos',
 'hao',
 'havemos',
 'haver',
 'hei',
 'houve',
 'houvemos',
 'houver',
 'houvera',
 'houveram',
 'houveramos',
 'houverao',
 'houverei',
 'houverem',
 'houveremos',
 'houveria',
 'houveriam',
 'h

In [9]:
"_".join(df["texto_limpo"].iloc[0].split())

''

In [10]:
def remover_stopwords(texto):
    tokens = texto.split()
    tokens = [token for token in tokens if token not in stopwords_pt]
    return tokens


df["tokens_sem_stopwords"] = df["texto_limpo"].apply(remover_stopwords)
df["texto_sem_stopwords"] = df["tokens_sem_stopwords"].str.join(" ")

df[["texto_limpo", "tokens_sem_stopwords", "texto_sem_stopwords"]].head()

,texto_limpo,tokens_sem_stopwords,texto_sem_stopwords
0,,[],
1,,[],
2,,[],
3,,[],
4,,[],


## 4. Bag of Words

No Bag of Words, cada linha representa uma notícia e cada coluna representa
uma palavra. O valor indica quantas vezes aquela palavra apareceu na
notícia.

Exemplo: `"motor turbo motor"` teria `motor = 2` e `turbo = 1`.

### Ajustando o vocabulário à escala da nossa base

A base do professor tinha 271 notícias. A nossa base tem cerca de 27 mil
notícias coletadas entre 2016 e 2026, o que gera um vocabulário muito maior
- por volta de 90 mil palavras distintas. Boa parte dessas palavras e
"cauda longa": nomes de modelos pouco citados, variacoes de escrita, erros
de digitacao e termos que aparecem em uma unica noticia. Elas nao ajudam a
generalizar padroes e custam memoria.

Se criarmos o `CountVectorizer` sem nenhum limite, ao chamar `.toarray()`
para transformar a matriz esparsa em uma matriz densa, o Python tenta
alocar uma celula de memoria para cada combinacao noticia x palavra do
vocabulario inteiro - incluindo os zeros, que sao a maioria. Com 27 mil
noticias e 90 mil palavras, isso ultrapassa 18 GB de memoria e o notebook
quebra com `MemoryError`.

Por isso, limitamos o vocabulario com tres parametros do
`CountVectorizer`, que tambem sao boa pratica em PLN (nao e so um ajuste
tecnico para caber na memoria):

- `min_df=5`: ignora palavras que aparecem em menos de 5 noticias - em
  geral ruido, erro de digitacao ou nome proprio muito especifico.
- `max_df=0.90`: ignora palavras que aparecem em mais de 90% das noticias
  - termos tao comuns que pouco ajudam a diferenciar um documento do
  outro.
- `max_features=5000`: dentro do que sobrar depois dos dois filtros
  acima, mantem apenas as 5000 palavras mais frequentes.

Tambem trocamos o tipo dos dados de `int64` (padrao) para `int32`: como as
contagens de palavras nao passam de alguns milhares, `int32` e suficiente
e usa metade da memoria.

In [11]:
from sklearn.feature_extraction.text import CountVectorizer
import numpy as np

vectorizer = CountVectorizer(
    min_df=5,
    max_df=0.90,
    max_features=5000,
    dtype=np.int32,
)
matriz_bow = vectorizer.fit_transform(df["texto_sem_stopwords"])

print(f"Vocabulario apos os filtros: {len(vectorizer.get_feature_names_out())} palavras")

df_bow = pd.DataFrame(
    matriz_bow.toarray(),
    columns=vectorizer.get_feature_names_out()
)

df_bow.head()

Vocabulario apos os filtros: 5000 palavras


,00,000,001,003,007,008,01,010,011,014,...,youtube,yuan,zeekr,zen,zero,zf,zoe,zona,zonas,zr
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [12]:
df["texto_completo"].iloc[1]

'Queremos a sua opinião! O que você gostaria de ver no Motor1.com? - Equipe do Motor1.com'

### Removendo colunas com números

Notícias de carros costumam trazer muitos números no texto (cilindradas,
preços, anos, potência em cv). Esses números geram colunas pouco úteis para
uma análise de tema (ex.: `1200`, `205`, `4x4`). Assim como no notebook de
referência, vamos remover qualquer coluna cujo nome tenha pelo menos um
número.

In [13]:
colunas_com_numeros = [col for col in df_bow.columns if any(char.isdigit() for char in col)]

df_bow = df_bow.drop(columns=colunas_com_numeros)

print(f"{len(colunas_com_numeros)} colunas removidas")
df_bow.head()

1077 colunas removidas


,abaixo,abandonar,abarth,abastecer,abastecido,abastecimento,abeifa,aberta,aberto,abertura,...,youtube,yuan,zeekr,zen,zero,zf,zoe,zona,zonas,zr
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


### Criando o DataFrame final

Agora vamos juntar os metadados das notícias com as colunas do Bag of Words.
Para evitar conflito de nomes, as colunas do Bag of Words recebem o prefixo
`bow_`.

In [14]:
metadados = df[["id_noticia", "data_publicacao", "ano", "categoria", "autor", "titulo", "url"]].reset_index(drop=True)
bow_com_prefixo = df_bow.add_prefix("bow_").reset_index(drop=True)

df_final = pd.concat([metadados, bow_com_prefixo], axis=1)

df_final.head()

,id_noticia,data_publicacao,ano,categoria,autor,titulo,url,bow_abaixo,bow_abandonar,bow_abarth,...,bow_youtube,bow_yuan,bow_zeekr,bow_zen,bow_zero,bow_zf,bow_zoe,bow_zona,bow_zonas,bow_zr
0,121597,2016-01-01 08:01:33,2016,Geral,Por:Redação,Faróis a laser: você está disposto a pagar (ca...,https://motor1.uol.com.br/news/121597/farois-a...,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,121653,2016-01-04 15:18:00,2016,Motos,Por:Redação2,Inusitada: BMW R1200 GS ganha versão com traçã...,https://motor1.uol.com.br/news/121653/inusitad...,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,121707,2016-01-05 10:00:00,2016,Tecnologia,Por:Redação,FCA apresenta central Uconnect com Android Aut...,https://motor1.uol.com.br/news/121707/fca-apre...,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,121745,2016-01-06 13:45:24,2016,Motos,Por:Redação,CES 2016: BMW apresenta capacete com head-up d...,https://motor1.uol.com.br/news/121745/ces-2016...,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,121766,2016-01-09 08:20:25,2016,Geral,Por:Dyogo Fagundes,Veja a lista dos 10 carros mais vendidos nos E...,https://motor1.uol.com.br/news/121766/veja-a-l...,0,0,0,...,0,0,0,0,0,0,0,0,0,0


### Removendo notícias sem conteúdo útil

Antes de seguir com a análise, verificamos se alguma notícia ficou sem
nenhuma palavra representada no vocabulário do Bag of Words. Isso pode
acontecer por dois motivos:

- o texto coletado no scraping era, na prática, só o widget de enquete do
  site (problema já visto em notícias antigas, de 2016, cujo layout era
  diferente do atual), e ficou vazio depois da remoção de boilerplate;
- todas as palavras da notícia foram filtradas pelos parâmetros
  `min_df`/`max_df`/`max_features` do `CountVectorizer`, definidos na
  seção anterior.

Se não tratarmos isso agora, o cálculo do TF mais adiante — que divide
pelo total de palavras de cada notícia — vai gerar uma divisão por zero
para essas notícias, resultando em `NaN` em toda a linha da matriz TF-IDF.
Por isso, removemos essas notícias aqui, de forma explícita e com o
diagnóstico impresso na tela, para que a análise exploratória, o TF-IDF
na mão e o TF-IDF com scikit-learn usem consistentemente o mesmo conjunto
de dados.

In [15]:
colunas_bow = [coluna for coluna in df_final.columns if coluna.startswith("bow_")]

noticias_sem_conteudo = df_final[colunas_bow].sum(axis=1) == 0
print(f"Removendo {noticias_sem_conteudo.sum()} noticias sem conteudo textual util no vocabulario do BoW")

df = df.loc[~noticias_sem_conteudo.values].reset_index(drop=True)
df_final = df_final.loc[~noticias_sem_conteudo].reset_index(drop=True)

print(f"Noticias restantes: {len(df_final)}")

Removendo 325 noticias sem conteudo textual util no vocabulario do BoW
Noticias restantes: 26592


## Exemplo de análise

Agora podemos fazer uma análise simples das palavras do Bag of Words: quais
aparecem mais, quais aparecem menos e quantas palavras diferentes temos no
vocabulário.

In [16]:
colunas_bow = [coluna for coluna in df_final.columns if coluna.startswith("bow_")]

frequencia_palavras = df_final[colunas_bow].sum().sort_values(ascending=False)

print(f"Total de palavras diferentes: {len(frequencia_palavras)}")

frequencia_palavras.head(10)

Total de palavras diferentes: 3923


bow_ainda     32819
bow_modelo    31827
bow_marca     31546
bow_brasil    29888
bow_motor     28835
bow_novo      27212
bow_versao    26972
bow_cv        26480
bow_fiat      23497
bow_nova      23073
dtype: int64

In [17]:
frequencia_palavras.tail(10)

bow_catarina    205
bow_fizemos     205
bow_perdido     205
bow_falam       205
bow_convite     205
bow_setores     205
bow_sv          205
bow_radar       205
bow_duplas      205
bow_abeifa      205
dtype: int64

In [18]:
documentos_por_palavra = (df_final[colunas_bow] > 0).sum().sort_values(ascending=False)
documentos_por_palavra.index = documentos_por_palavra.index.str.replace("bow_", "", regex=False)

documentos_por_palavra.head(10)

ainda     16883
marca     16550
modelo    15690
brasil    14280
motor     14093
versao    12863
novo      12448
cv        12429
nova      12118
ano       12071
dtype: int64

In [19]:
df_final["palavras_unicas"] = (df_final[colunas_bow] > 0).sum(axis=1)

df_final[["titulo", "palavras_unicas"]].sort_values("palavras_unicas", ascending=True).head(10)

,titulo,palavras_unicas
21321,"F1 - Ferrari: ""Tudo deu errado"" no Canadá, mas...",1
1752,"Nos 23 anos sem Senna, veja 23 grandes momento...",12
24040,JAC Hunter 2026 ganha versão 4Work mais barata...,15
18287,Motor1.com Experience: Quer participar de uma ...,18
2948,Vídeo: Nelson Piquet fala sobre carros e prefe...,19
4529,Transplante bem sucedido: 10 carros que mudara...,20
10,"Vídeo: motorista de ônibus furioso ""empurra"" t...",21
774,Vídeo: Troller T4 empurra Fiat Idea que atrapa...,22
154,Galeria futurista: os carros da McLaren e Toro...,24
4468,Quero ser SUV: 10 carros que pegaram carona no...,24


## 5. TF-IDF

O Bag of Words transforma textos em números contando quantas vezes cada
palavra aparece em cada notícia. Essa representação é simples e útil, mas
tem uma limitação importante: palavras muito frequentes podem parecer
importantes apenas porque aparecem muito.

O **TF-IDF** é uma forma de dar peso para as palavras tentando responder a
duas perguntas ao mesmo tempo:

- Esta palavra aparece bastante nesta notícia?
- Esta palavra é rara no conjunto de notícias?

A ideia é que uma palavra deve receber peso alto quando aparece bastante em
uma notícia, mas não aparece em todas as notícias. Assim, o TF-IDF ajuda a
destacar palavras mais características de cada documento — por exemplo,
"elétrico" em notícias sobre eletrificação, ou "moto" em notícias da
categoria Motos.

### TF: Term Frequency

**TF** significa *Term Frequency*, ou frequência do termo.

Ele mede a importância de uma palavra dentro de uma notícia específica. Se
uma palavra aparece muitas vezes em uma notícia, o TF dela naquela notícia
será maior.

$$TF(t, d) = \frac{\text{quantidade do termo } t \text{ no documento } d}{\text{total de termos no documento } d}$$

Exemplo: se uma notícia tem 100 palavras e a palavra `motor` aparece 5
vezes, então:

$$TF(\text{motor}) = \frac{5}{100} = 0.05$$

O TF olha apenas para dentro de uma notícia. Ele ainda não sabe se a
palavra é comum ou rara no conjunto inteiro.

### IDF: Inverse Document Frequency

**IDF** significa *Inverse Document Frequency*, ou frequência inversa nos
documentos.

Ele mede o quanto uma palavra é rara no conjunto de notícias. Palavras que
aparecem em muitas notícias recebem peso menor. Palavras que aparecem em
poucas notícias recebem peso maior.

$$IDF(t) = \log\left(\frac{N}{DF(t)}\right)$$

Onde:

- $N$ é o número total de notícias.
- $DF(t)$ é o número de notícias em que o termo $t$ aparece.

Se uma palavra aparece em quase todas as notícias — como pode acontecer com
"carro" em um site de notícias automotivas — ela não ajuda muito a
diferenciar uma notícia da outra. Por isso, seu IDF fica baixo.

### TF-IDF

O **TF-IDF** combina as duas ideias:

$$TF\text{-}IDF(t, d) = TF(t, d) \times IDF(t)$$

Uma palavra terá peso alto quando:

- aparece bastante em uma notícia;
- não aparece em muitas outras notícias.

Por isso, TF-IDF costuma ser melhor do que uma contagem simples quando
queremos comparar notícias, encontrar palavras importantes de cada matéria
ou preparar textos para modelos de aprendizado de máquina.

## Como calcular TF-IDF na mão

Como já criamos o `df_final`, temos uma matriz Bag of Words pronta nas
colunas que começam com `bow_`.

Para calcular o TF-IDF manualmente com pandas, o caminho é:

1. Separar apenas as colunas `bow_` do `df_final`.
2. Remover o prefixo `bow_` dos nomes das colunas, para trabalhar
   diretamente com as palavras.
3. Calcular o total de palavras de cada notícia somando as linhas da
   matriz.
4. Calcular o **TF** dividindo cada valor da linha pelo total da própria
   linha.
5. Calcular em quantas notícias cada palavra aparece.
6. Calcular o **IDF** usando $\log(N / DF)$.
7. Multiplicar a matriz de TF pelo vetor de IDF.

O resultado final é uma nova matriz: cada linha continua sendo uma notícia,
cada coluna continua sendo uma palavra, mas agora os valores são pesos
TF-IDF.

## Implementação

Agora vamos implementar. A ideia é preencher as próximas células seguindo
os passos explicados acima.

In [20]:
matriz_bow_df = df_final[colunas_bow].copy()
matriz_bow_df.columns = matriz_bow_df.columns.str.replace("bow_", "", regex=False)

matriz_bow_df.head()

,abaixo,abandonar,abarth,abastecer,abastecido,abastecimento,abeifa,aberta,aberto,abertura,...,youtube,yuan,zeekr,zen,zero,zf,zoe,zona,zonas,zr
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,1,2,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,1,...,0,0,0,0,0,1,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,1,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0


In [21]:
total_palavras_por_noticia = matriz_bow_df.sum(axis=1)

matriz_tf = matriz_bow_df.div(total_palavras_por_noticia, axis=0)

matriz_tf.head()

,abaixo,abandonar,abarth,abastecer,abastecido,abastecimento,abeifa,aberta,aberto,abertura,...,youtube,yuan,zeekr,zen,zero,zf,zoe,zona,zonas,zr
0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.000000,0.000000,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.00000,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.001859,0.003717,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.00000,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.000000,0.000000,0.001170,...,0.0,0.0,0.0,0.0,0.0,0.00117,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.000000,0.000000,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.00000,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.001506,0.0,0.0,0.000000,0.000000,0.001506,...,0.0,0.0,0.0,0.0,0.0,0.00000,0.0,0.0,0.0,0.0


In [22]:
noticia = 0

print(df_final.loc[noticia, "titulo"])

matriz_tf.loc[noticia].sort_values(ascending=False).head(10)

Flagra: Volkswagen Ameo aparece "limpo" pela primeira vez


vw              0.050000
exclusivo       0.033333
novo            0.033333
flagra          0.033333
polo            0.033333
seda            0.033333
especulacoes    0.016667
divulgacao      0.016667
feita           0.016667
camuflagem      0.016667
Name: 0, dtype: float64

In [23]:
import numpy as np

N = len(matriz_bow_df)
documentos_por_termo = (matriz_bow_df > 0).sum(axis=0)

idf = np.log(N / documentos_por_termo)

df_idf = pd.DataFrame({
    "documentos_com_o_termo": documentos_por_termo,
    "idf": idf
}).sort_values("idf", ascending=False)

df_idf.head(10)

,documentos_com_o_termo,idf
su,38,6.550780
con,41,6.474794
una,44,6.404176
en,44,6.404176
un,45,6.381703
steering,64,6.029483
vencimento,66,5.998711
vettel,70,5.939870
himalayan,70,5.939870
hornet,71,5.925686


In [24]:
df_idf.tail(10)

,documentos_com_o_termo,idf
ano,12071,0.789805
nova,12118,0.785918
cv,12429,0.760578
novo,12448,0.759050
versao,12863,0.726255
motor,14093,0.634932
brasil,14280,0.621750
modelo,15690,0.527587
marca,16550,0.474224
ainda,16883,0.454303


In [25]:
matriz_tfidf = matriz_tf.mul(idf, axis=1)

matriz_tfidf.head()

,abaixo,abandonar,abarth,abastecer,abastecido,abastecimento,abeifa,aberta,aberto,abertura,...,youtube,yuan,zeekr,zen,zero,zf,zoe,zona,zonas,zr
0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.000000,0.000000,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.008609,0.017105,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.000000,0.000000,0.003611,...,0.0,0.0,0.0,0.0,0.0,0.006269,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.000000,0.000000,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.005683,0.0,0.0,0.000000,0.000000,0.004650,...,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0


In [26]:
noticia = 10

print(df_final.loc[noticia, "titulo"])

matriz_tfidf.loc[noticia].sort_values(ascending=False).head(10)

Vídeo: motorista de ônibus furioso "empurra" táxi no meio da rua no Rio de Janeiro


transito          0.329629
jogos             0.240478
comuns            0.203093
dessas            0.202945
realizacao        0.202501
impressionante    0.194435
ninguem           0.190542
rio               0.170848
spin              0.169192
dizer             0.142647
Name: 10, dtype: float64

In [27]:
tfidf_com_prefixo = matriz_tfidf.add_prefix("tfidf_").reset_index(drop=True)

df_final_tfidf = pd.concat([metadados, tfidf_com_prefixo], axis=1)

df_final_tfidf.head()

,id_noticia,data_publicacao,ano,categoria,autor,titulo,url,tfidf_abaixo,tfidf_abandonar,tfidf_abarth,...,tfidf_youtube,tfidf_yuan,tfidf_zeekr,tfidf_zen,tfidf_zero,tfidf_zf,tfidf_zoe,tfidf_zona,tfidf_zonas,tfidf_zr
0,121597,2016-01-01 08:01:33,2016,Geral,Por:Redação,Faróis a laser: você está disposto a pagar (ca...,https://motor1.uol.com.br/news/121597/farois-a...,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0
1,121653,2016-01-04 15:18:00,2016,Motos,Por:Redação2,Inusitada: BMW R1200 GS ganha versão com traçã...,https://motor1.uol.com.br/news/121653/inusitad...,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0
2,121707,2016-01-05 10:00:00,2016,Tecnologia,Por:Redação,FCA apresenta central Uconnect com Android Aut...,https://motor1.uol.com.br/news/121707/fca-apre...,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.006269,0.0,0.0,0.0,0.0
3,121745,2016-01-06 13:45:24,2016,Motos,Por:Redação,CES 2016: BMW apresenta capacete com head-up d...,https://motor1.uol.com.br/news/121745/ces-2016...,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0
4,121766,2016-01-09 08:20:25,2016,Geral,Por:Dyogo Fagundes,Veja a lista dos 10 carros mais vendidos nos E...,https://motor1.uol.com.br/news/121766/veja-a-l...,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0


## Como calcular TF-IDF com scikit-learn

Na prática, também podemos usar o `TfidfVectorizer` do scikit-learn.

Ele faz em uma única etapa o que fizemos manualmente:

- cria o vocabulário;
- conta os termos;
- calcula os pesos TF-IDF;
- devolve uma matriz numérica.

Como nossos textos já foram limpos anteriormente, podemos aplicar o
`TfidfVectorizer` diretamente em `df["texto_sem_stopwords"]`.

A vantagem do scikit-learn é que ele já resolve detalhes de implementação e
devolve uma matriz pronta para ser usada em modelos de aprendizado de
máquina. A vantagem de fazer na mão primeiro é entender o que está
acontecendo por baixo.

Pelo mesmo motivo explicado na seção de Bag of Words, aplicamos os mesmos
limites de vocabulário (`min_df`, `max_df`, `max_features`) ao
`TfidfVectorizer`: sem eles, `.toarray()` tentaria alocar uma matriz densa
de mais de 18 GB para o nosso corpus de ~27 mil notícias e o notebook
quebraria com `MemoryError`, do mesmo jeito que aconteceu no
`CountVectorizer`.

In [28]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer_tfidf = TfidfVectorizer(
    min_df=5,
    max_df=0.90,
    max_features=5000,
)
matriz_tfidf_sklearn = vectorizer_tfidf.fit_transform(df["texto_sem_stopwords"])

df_tfidf_sklearn = pd.DataFrame(
    matriz_tfidf_sklearn.toarray(),
    columns=vectorizer_tfidf.get_feature_names_out()
)

df_tfidf_sklearn.head()

,00,000,001,003,007,008,01,010,011,014,...,youtube,yuan,zeekr,zen,zero,zf,zoe,zona,zonas,zr
0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0
1,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0
2,0.0,0.013189,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.030634,0.0,0.0,0.0,0.0
3,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0
4,0.0,0.016181,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0


## Salvando os dados

Vamos salvar as matrizes Bag of Words e TF-IDF em arquivos CSV para
reaproveitar em outras análises da próxima etapa do projeto (classificação
supervisionada por `categoria`).

In [ ]:
# Dados completos: metadados + texto original + texto limpo + texto sem stopwords
df_completo = df[[
    "id_noticia", "url", "titulo", "subtitulo", "categoria", "autor",
    "data_publicacao", "ano", "mes",
    "texto_completo", "texto_sem_boilerplate", "texto_limpo", "texto_sem_stopwords",
]]
df_completo.to_csv(pasta_dados / "noticias_motor1_processado.csv", index=False)

# Bag of Words (metadados + colunas bow_)
df_final.to_csv(pasta_dados / "bow_motor1.csv", index=False)

# TF-IDF (metadados + colunas tfidf_)
df_final_tfidf.to_csv(pasta_dados / "tfidf_motor1.csv", index=False)

print(f"Noticias processadas: {pasta_dados / 'noticias_motor1_processado.csv'} (shape={df_completo.shape})")
print(f"Bag of Words:         {pasta_dados / 'bow_motor1.csv'}      (shape={df_final.shape})")
print(f"TF-IDF:               {pasta_dados / 'tfidf_motor1.csv'}    (shape={df_final_tfidf.shape})")